# Chapter 2: Working with Text Data

Full pipeline: **Raw Text → Tokenize → Embeddings → DataLoader → Training Batch**

## 1. Load the Sample Text

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text)}")
print(f"First 200 chars:\n{raw_text[:200]}")

## 2. Simple Word-Level Tokenizer

Let's first build a basic tokenizer from scratch to understand the concept.

In [ ]:
from tokenizer import SimpleTokenizer, build_vocab

# Build vocabulary from our text
vocab = build_vocab(raw_text)
print(f"Vocabulary size: {len(vocab)}")

# Show some vocabulary entries
for token, idx in list(vocab.items())[:10]:
    print(f"  '{token}' → {idx}")

In [ ]:
# Encode and decode a snippet
tokenizer = SimpleTokenizer(vocab)

snippet = '"It\'s the last day," she said.'
print(f"Original: {snippet}")

ids = tokenizer.encode(snippet)
print(f"Encoded:  {ids}")

decoded = tokenizer.decode(ids)
print(f"Decoded:  {decoded}")

**Limitation:** This tokenizer can only handle words that appear in the training text. Unknown words cause a `KeyError`.

## 3. BPE Tokenizer (tiktoken)

GPT-2 uses Byte Pair Encoding — it can handle *any* text.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print(f"GPT-2 vocabulary size: {enc.n_vocab}")

# Encode sample text
sample = "Hello, do you like tea? <|endoftext|> In the sunlit terraces."
token_ids = enc.encode(sample, allowed_special={"<|endoftext|>"})
print(f"\nText: {sample}")
print(f"Token IDs: {token_ids}")
print(f"Num tokens: {len(token_ids)}")

# Decode back
print(f"Decoded: {enc.decode(token_ids)}")

In [ ]:
# Tokenize the full text
all_tokens = enc.encode(raw_text)
print(f"The Verdict: {len(raw_text)} characters → {len(all_tokens)} BPE tokens")
print(f"Average chars per token: {len(raw_text) / len(all_tokens):.1f}")

## 4. Token + Positional Embeddings

Now we convert token IDs into dense vectors the model can learn from.

In [ ]:
import torch
from embeddings import TokenPositionalEmbedding

# GPT-2 style config
VOCAB_SIZE = 50257
EMBED_DIM = 256
MAX_LEN = 1024

# Create embedding layer
embedding = TokenPositionalEmbedding(VOCAB_SIZE, EMBED_DIM, MAX_LEN)
print(f"Embedding layer: vocab={VOCAB_SIZE}, dim={EMBED_DIM}, max_len={MAX_LEN}")

# Embed a small sequence
sample_ids = enc.encode("Every journey begins")
x = torch.tensor([sample_ids])  # batch of 1
print(f"\nInput shape:  {x.shape}  (batch_size, seq_len)")

out = embedding(x)
print(f"Output shape: {out.shape}  (batch_size, seq_len, embed_dim)")
print(f"\nEach token is now a {EMBED_DIM}-dimensional vector.")
print(f"First token vector (first 8 dims): {out[0, 0, :8].detach().tolist()}")

## 5. Data Loader — Sliding Window

Create input-target pairs for next-token prediction training.

In [ ]:
from dataloader import create_dataloader

# Small context window to see the pattern clearly
loader = create_dataloader(
    raw_text, enc,
    max_len=4, stride=1, batch_size=8, shuffle=False
)

inputs, targets = next(iter(loader))
print("Sliding window with max_len=4, stride=1:")
print(f"Batch shape: inputs={inputs.shape}, targets={targets.shape}\n")

for i in range(4):
    inp = inputs[i].tolist()
    tgt = targets[i].tolist()
    print(f"Sample {i}: input={inp} → target={tgt}")
    print(f"           '{enc.decode(inp)}' → '{enc.decode(tgt)}'")

Notice how the target is just the input shifted right by 1 position. This is next-token prediction!

## 6. Full Pipeline — One Training Batch

Putting it all together: text → tokens → embeddings → ready for the model.

In [ ]:
# Realistic config
MAX_LEN = 256
STRIDE = 256
BATCH_SIZE = 2

# Create dataloader
train_loader = create_dataloader(
    raw_text, enc,
    max_len=MAX_LEN, stride=STRIDE, batch_size=BATCH_SIZE
)

print(f"Dataset: {len(train_loader.dataset)} samples")
print(f"Batches: {len(train_loader)}")

# Get one batch
inputs, targets = next(iter(train_loader))
print(f"\nBatch tensors:")
print(f"  inputs:  {inputs.shape}  (batch_size, seq_len)")
print(f"  targets: {targets.shape}  (batch_size, seq_len)")

# Pass through embedding layer
embedded = embedding(inputs)
print(f"  embedded: {embedded.shape}  (batch_size, seq_len, embed_dim)")

print(f"\n✅ This embedded tensor is what goes into the transformer layers!")

## Summary

```
Raw Text
  ↓  tokenizer (BPE)
Token IDs  [15496, 11, 466, ...]
  ↓  nn.Embedding
Token Embeddings  (batch, seq_len, embed_dim)
  +
Positional Embeddings  (seq_len, embed_dim)
  =
Input Embeddings  → ready for transformer layers
```

**Next chapter:** Attention mechanisms — how the model learns to look at relevant tokens.